In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
from scipy.stats import wilcoxon
warnings.filterwarnings('ignore')

CSV_PATH = 'recorded-data-final.csv'
METRICS = {
    'UND': 'Understandability',
    'TRU': 'Trustworthiness',
    'INS': 'Insightfulness',
    'SAT': 'Satisfaction',
    'CON': 'Confidence',
    'CVN': 'Convincingness',
    'COM': 'Communicability',
    'USB': 'Usability'
}
METRIC_ORDER = list(METRICS.keys())
DEEPSEEK = 'Model_2'
GEMINI   = 'Model_3'

CRPS_QUESTION_MAP = {
    'Q135': 'UND', 'Q136': 'UND', 'Q138': 'UND', 'Q140': 'UND',
    'Q141': 'TRU', 'Q142': 'TRU', 'Q143': 'TRU', 'Q144': 'TRU',
    'Q145': 'INS', 'Q146': 'INS', 'Q147': 'INS', 'Q148': 'INS',
    'Q149': 'SAT', 'Q150': 'SAT', 'Q151': 'SAT', 'Q152': 'SAT',
    'Q153': 'CON', 'Q155': 'CON', 'Q154': 'CON', 'Q156': 'CON',
    'Q157': 'CVN', 'Q158': 'CVN', 'Q159': 'CVN', 'Q160': 'CVN',
    'Q161': 'COM', 'Q162': 'COM', 'Q163': 'COM', 'Q164': 'COM',
    'Q165': 'USB', 'Q166': 'USB', 'Q167': 'USB', 'Q168': 'USB'
}
NCRPS_QUESTION_MAP = {
    'NPA_1': 'UND', 'Q114': 'TRU', 'Q116': 'INS', 'Q117': 'SAT',
    'Q118': 'CON', 'Q119': 'CVN', 'Q120': 'COM', 'Q121': 'USB',
    'Q175': 'UND', 'Q176': 'UND', 'Q177': 'UND',
    'Q178': 'TRU', 'Q179': 'TRU', 'Q180': 'TRU',
    'Q181': 'INS', 'Q182': 'INS', 'Q183': 'INS',
    'Q184': 'SAT', 'Q185': 'SAT', 'Q186': 'SAT',
    'Q187': 'CON', 'Q188': 'CON', 'Q189': 'CON',
    'Q190': 'CVN', 'Q191': 'CVN', 'Q192': 'CVN',
    'Q194': 'COM', 'Q195': 'COM', 'Q196': 'COM',
    'Q197': 'USB', 'Q199': 'USB', 'Q201': 'USB'
}

In [2]:
NON_ANSWERS = {
    'unsure', 'prefer not to answer', 'i am not sure', "i don't know",
    'idk', 'n/a', 'na', 'not applicable', 'skip', 'blank'
}
LIKERT_WORDS = ('agree', 'disagree', 'neutral', 'strongly', 'slightly')

def _looks_like_nonanswer(s):
    return any(tok in str(s).lower().strip() for tok in NON_ANSWERS)

def _extract_digit_candidates(s):
    cands = re.findall(r'\((\s*[1-5]\s*)\)', str(s))
    m = re.match(r'^\s*([1-5])\s*(?:[-\u2013\u2014]|\b)', str(s))
    if m:
        cands.append(m.group(1))
    if any(w in str(s).lower() for w in LIKERT_WORDS):
        cands += re.findall(r'(?<!\d)([1-5])(?!\d)', str(s))
    cands = [re.sub(r'\s+', '', d) for d in cands]
    cands = [d for d in cands if d in {'1','2','3','4','5'}]
    seen, uniq = set(), []
    for d in cands:
        if d not in seen:
            seen.add(d); uniq.append(d)
    return uniq

def likert(val):
    if pd.isna(val): return np.nan
    s = str(val).strip()
    if s.startswith('{"ImportId"'): return np.nan
    if _looks_like_nonanswer(s): return np.nan
    cands = _extract_digit_candidates(s)
    if len(cands) == 1: return float(cands[0])
    if re.fullmatch(r'[1-5]', s): return float(s)
    return np.nan

def to_cohort(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().lower()
    if s.startswith('yes') or s in {'y','1','true','crps'}: return 'CRPS'
    if s.startswith('no')  or s in {'n','0','false','ncrps'}: return 'NCRPS'
    return np.nan

In [ ]:
df = pd.read_csv(CSV_PATH, skiprows=[1])
df['cohort'] = df['Q92'].apply(to_cohort)
df['participant_id'] = df.get('ResponseId', pd.RangeIndex(len(df)))

print(f'Loaded {len(df)} responses')
print(df['cohort'].value_counts(dropna=False))

long_records = []
for _, row in df.iterrows():
    pid    = row['participant_id']
    cohort = row['cohort']
    if pd.isna(cohort): continue
    qmap = CRPS_QUESTION_MAP if cohort == 'CRPS' else NCRPS_QUESTION_MAP
    for loop in (1, 2, 3):
        model = f'Model_{loop}'
        for qid, metric in qmap.items():
            col = f'{loop}_{qid}'
            if col in df.columns:
                sc = likert(row[col])
                if pd.notna(sc) and 1 <= sc <= 5:
                    long_records.append({
                        'participant_id': pid,
                        'cohort': cohort,
                        'model': model,
                        'metric': metric,
                        'score': sc
                    })

df_long = pd.DataFrame(long_records)
print(f'\nLong format: {len(df_long):,} observations, {df_long["participant_id"].nunique()} participants')

In [ ]:
pmeans = (
    df_long
    .groupby(['participant_id', 'cohort', 'model', 'metric'])['score']
    .mean()
    .reset_index()
)
print(f'Participant-level means: {len(pmeans)} rows')

In [5]:
def holm(p_values):

    p = np.asarray(p_values, dtype=float)
    m = len(p)
    order = np.argsort(p)
    adj = np.zeros(m)
    running_max = 0.0
    for rank, idx in enumerate(order):
        corrected = p[idx] * (m - rank)
        running_max = max(running_max, corrected)
        adj[idx] = min(running_max, 1.0)
    return adj

def bh(p_values):

    p = np.asarray(p_values, dtype=float)
    m = len(p)
    order = np.argsort(p)
    q = np.zeros(m)
    for i in range(m - 1, -1, -1):
        idx = order[i]
        if i == m - 1:
            q[idx] = p[idx]
        else:
            q[idx] = min(p[idx] * m / (i + 1), q[order[i + 1]])
    return q

In [ ]:
raw_results = []

for cohort in ['CRPS', 'NCRPS']:
    cohort_data = pmeans[pmeans['cohort'] == cohort]

    ds_data  = cohort_data[cohort_data['model'] == DEEPSEEK][['participant_id', 'metric', 'score']]
    gem_data = cohort_data[cohort_data['model'] == GEMINI ][['participant_id', 'metric', 'score']]
    paired = ds_data.merge(gem_data, on=['participant_id', 'metric'],suffixes=('_ds', '_gem'))
    paired['diff'] = paired['score_ds'] - paired['score_gem']

    for metric in METRIC_ORDER:
        diffs = paired[paired['metric'] == metric]['diff'].dropna().values

        if len(diffs) < 3:
            print(f'WARNING: {cohort}/{metric} — only {len(diffs)} paired obs, skipping')
            continue

        mean_diff = diffs.mean()
        sd_diff   = diffs.std(ddof=1)
        n_pairs   = len(diffs)

        if np.all(diffs == 0):
            stat, p_raw = np.nan, 1.0
        else:
            stat, p_raw = wilcoxon(diffs, alternative='two-sided')

        raw_results.append({
            'cohort':    cohort,
            'metric':    metric,
            'n_pairs':   n_pairs,
            'mean_diff': mean_diff,
            'sd_diff':   sd_diff,
            'stat':      stat,
            'p_raw':     p_raw,
        })

results_df = pd.DataFrame(raw_results)
print(f'Computed {len(results_df)} paired tests')
print(results_df[['cohort','metric','n_pairs','mean_diff','sd_diff','p_raw']].to_string(index=False))

In [ ]:
for cohort in ['CRPS', 'NCRPS']:
    mask = results_df['cohort'] == cohort
    p_raw = results_df.loc[mask, 'p_raw'].values
    results_df.loc[mask, 'p_holm'] = holm(p_raw)
    results_df.loc[mask, 'p_bh']   = bh(p_raw)

results_df['p_holm'] = results_df['p_holm'].clip(upper=1.0)
results_df['p_bh']   = results_df['p_bh'].clip(upper=1.0)

print('Corrections applied (m=8 per cohort)')
print(results_df[['cohort','metric','p_raw','p_holm','p_bh']].round(4).to_string(index=False))

In [ ]:
out = results_df[['cohort','metric','n_pairs','mean_diff','sd_diff','p_raw','p_holm','p_bh']].copy()
out = out.round({'mean_diff': 3, 'sd_diff': 3, 'p_raw': 6, 'p_holm': 4, 'p_bh': 4})
print(out.to_string(index=False))